# 学习率调度器

## 1.Warmup
> **为什么一定要有 Warmup (预热)？**
> 1. **模型随机初始化**时，梯度非常大规模且方向混乱。如果直接给最大的学习率（如 3e-4），大规模的梯度更新会瞬间把模型权重冲飞 (Loss 直接 NaN)。
> 2. **AdamW 优化器**在刚开始时，其用于分母的“二阶动量 (方差的移动平均)”还没收集够数据，非常小。除以一个极小的数会导致实际更新步长不可控。Warmup 给了优化器几千步的“收集方差”的时间。

## 2. Cosine Annealing with Warmup (带预热的余弦退火)
学习率从 base_lr 平滑下降到 min_lr，曲线像余弦函数的前半段
公式：

$$
\mathrm{lr}(t) = \mathrm{lr}_{\min} + \tfrac{1}{2}(\mathrm{lr}_{\max}-\mathrm{lr}_{\min})\left(1+\cos(\pi p)\right)
$$

其中：

$$p=\frac{t}{T}$$

- $t$: 当前 decay 阶段已走的步数；
- $T$: 总 decay 步数；
- $p$: 进度，范围通常是 0~1；
- $lr_{\max}$: base_lr；
- $lr_{\min}$: 最低学习率。

## 3.WSD (Warmup-Stable-Decay)

1. **Warmup (预热)：** 线性增长到最大学习率。
2. **Stable (稳定期)：** 保持最大学习率不变，训练绝大部分 Token（占整体的 70%~90%）。吃尽海量数据。如果想加数据，无限延长这个阶段即可。
3. **Decay (高效退火)：** 只在训练的最后 10% 或 5% 阶段，用一个陡峭的函数（如线性或余弦）快速降到 0，让模型迅速收敛收拢。

```
    lr
    │                     ┌──────────── Stable ────────────┐
    │                  ┌──┘                                  \
    │               ┌──┘                                      \
    │            ┌──┘                                          \__
    │         ┌──┘                                                \___ min_lr
    │      ┌──┘
    │   ┌──┘
    └────────────────────────────────────────────────────────────── step
    Warmup                    Stable                 Cosine Decay
```

In [1]:
import torch
import math
import matplotlib.pyplot as plt
from torch.optim.lr_scheduler import LRScheduler

In [2]:
class WSD_Scheduler(LRScheduler):
    """
    手动实现 LLaMA-3 风格的 Warmup-Stable-Decay (WSD) 学习率调度器。
    """
    def __init__(self, optimizer, num_warmup_steps, num_stable_steps, num_decay_steps, min_lr_ratio=0.1, last_epoch=-1):
        self.num_warmup_steps = num_warmup_steps
        self.num_stable_steps = num_stable_steps
        self.num_decay_steps = num_decay_steps
        self.min_lr_ratio = min_lr_ratio
        self.total_steps = num_warmup_steps + num_stable_steps + num_decay_steps
        super().__init__(optimizer, last_epoch)
        
    def get_lr(self):
        step = self._step_count - 1
        
        lrs = []
        for base_lr in self.base_lrs:
            min_lr = base_lr * self.min_lr_ratio
            
            # ==========================================
            # TODO 1: Warmup 阶段
            # 规则: 当 step < num_warmup_steps 时，学习率从 0 线性增长到 base_lr
            # ==========================================
            if step < self.num_warmup_steps:
                if step == 0:
                    current_lr = 0.0
                else:
                    current_lr = base_lr * step / self.num_warmup_steps
            
            # ==========================================
            # TODO 2: Stable 阶段
            # 规则: 学习率保持在 base_lr
            # ==========================================
            elif step < (self.num_warmup_steps + self.num_stable_steps):
                current_lr = base_lr
                
            # ==========================================
            # 退火期按进度把学习率从 base_lr 拉到 min_lr。
            # TODO 3: Cosine Decay 阶段
            # 规则: 学习率从 base_lr 余弦衰减到 min_lr
            # 提示: 计算 decay 阶段的进度比例，使用余弦函数
            # ==========================================
            else:
                decay_step = step - self.num_warmup_steps - self.num_stable_steps
                decay_ratio = decay_step / self.num_decay_steps
                cosine_decay = 0.5 * (1 + math.cos(math.pi * decay_ratio))
                current_lr = min_lr + (base_lr - min_lr) * cosine_decay
                
            lrs.append(current_lr)
            
        return lrs

In [3]:
# 测试并可视化你的实现
def test_and_plot_wsd():
    try:
        # 1. 初始化一个假的优化器 (用来承载学习率)
        dummy_model = torch.nn.Linear(2, 2)
        max_lr = 3e-4
        optimizer = torch.optim.AdamW(dummy_model.parameters(), lr=max_lr)

        # 2. 设定 WSD 的三个阶段步数
        warmup = 1000   # 10%
        stable = 7000   # 70%
        decay = 2000    # 20%
        total = warmup + stable + decay

        # 3. 初始化我们实现的 Scheduler
        scheduler = WSD_Scheduler(
            optimizer,
            num_warmup_steps=warmup,
            num_stable_steps=stable,
            num_decay_steps=decay,
            min_lr_ratio=0.1
        )

        # 4. 模拟训练过程，收集学习率
        lrs = []
        for _ in range(total):
            lrs.append(optimizer.param_groups[0]['lr'])
            optimizer.step()
            scheduler.step()

        # 5. 断言关键点的正确性
        assert lrs[0] == 0.0, "第一步应该是 0 (或者极小值)"
        assert abs(lrs[warmup] - max_lr) < 1e-8, "Warmup 结束时应该是最大学习率"
        assert abs(lrs[warmup + stable - 1] - max_lr) < 1e-8, "Stable 阶段应该维持最大学习率"
        assert abs(lrs[-1] - (max_lr * 0.1)) < 1e-8, "Decay 结束时应该是最小学习率 (max_lr * 0.1)"

        print("✅ 数学逻辑断言通过！")

        # 6. 画出学习率曲线
        plt.figure(figsize=(10, 5))
        plt.plot(lrs, label="Learning Rate", color='blue', linewidth=2)
        plt.axvline(x=warmup, color='r', linestyle='--', alpha=0.5, label='End Warmup')
        plt.axvline(x=warmup + stable, color='g', linestyle='--', alpha=0.5, label='Start Decay')
        plt.title("LLaMA-3 Style WSD (Warmup-Stable-Decay) Scheduler")
        plt.xlabel("Training Steps")
        plt.ylabel("Learning Rate")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.close()

        print(" 你成功实现并可视化了目前最先进的大模型学习率调度器。现在你不怕被面试官问到 LLaMA-3 的退火策略了！")

    except NotImplementedError:
        print("请先完成 TODO 代码！")
        raise
    except (AttributeError, NameError, TypeError, ValueError) as e:
        print("代码可能未完成，导致变量未定义" if isinstance(e, NameError) else "代码可能未完成，导致了类型错误")
        raise NotImplementedError("请先完成 TODO 代码！") from e
    except AssertionError as e:
        print(f"❌ 测试失败: {e}")
        raise NotImplementedError("请先完成 TODO 代码！") from e
    except Exception as e:
        print(f"❌ 测试失败: {e}")
        raise

test_and_plot_wsd()

✅ 数学逻辑断言通过！
 你成功实现并可视化了目前最先进的大模型学习率调度器。现在你不怕被面试官问到 LLaMA-3 的退火策略了！
